**1.** Какая из причин отмены рейса (`CancellationCode`) была самой частой?

CancellationCode reason for cancellation (A = carrier, B = weather, C = NAS, D = security)

In [30]:
import pandas as pd
data = pd.read_csv('2008.csv')
print(data['CancellationCode'].describe())
print(data['CancellationCode'].value_counts().idxmax())

count     1411
unique       3
top          A
freq       563
Name: CancellationCode, dtype: object
A


In [44]:
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               70000 non-null  int64  
 1   Month              70000 non-null  int64  
 2   DayofMonth         70000 non-null  int64  
 3   DayOfWeek          70000 non-null  int64  
 4   DepTime            68601 non-null  float64
 5   CRSDepTime         70000 non-null  int64  
 6   ArrTime            68444 non-null  float64
 7   CRSArrTime         70000 non-null  int64  
 8   UniqueCarrier      70000 non-null  str    
 9   FlightNum          70000 non-null  int64  
 10  TailNum            69180 non-null  str    
 11  ActualElapsedTime  68418 non-null  float64
 12  CRSElapsedTime     69991 non-null  float64
 13  AirTime            68418 non-null  float64
 14  ArrDelay           68418 non-null  float64
 15  DepDelay           68601 non-null  float64
 16  Origin             70000 non-null

**2.** Найдите среднее, минимальное и максимальное расстояние, пройденное самолетом.

In [52]:
distance = data['Distance']
print(distance.mean(), distance.min(), distance.max(), sep='\n')

724.5082571428571
31
4962


**3.** Не выглядит ли подозрительным минимальное пройденное расстояние? В какие дни и на каких рейсах оно было? Какое расстояние было пройдено этими же рейсами в другие дни?

In [99]:
import numpy as np
min_distance_data = data.loc[data['Distance'] == distance.min(), ['Distance', 'TailNum', 'FlightNum', 'Month', 'DayofMonth', 'Origin', 'Dest']]
print(min_distance_data)

bad_tails = min_distance_data['TailNum']

in_other_days = data[data['TailNum'].isin(bad_tails)].groupby('TailNum')['Distance'].describe()
print(in_other_days)


       Distance TailNum  FlightNum  Month  DayofMonth Origin Dest
1116         31  N795AS         65     12          30    WRG  PSG
6958         31  N795AS         65     12          26    WRG  PSG
17349        31  N768AS         64      8          18    PSG  WRG
27534        31  N764AS         64      3          11    PSG  WRG
46082        31  N708AS         65      8           9    WRG  PSG
48112        31  N762AS         64      2          28    PSG  WRG
         count        mean         std   min    25%    50%      75%     max
TailNum                                                                    
N708AS    10.0  660.500000  387.046150  31.0  383.5  752.0   995.75  1050.0
N762AS    27.0  461.666667  389.320117  31.0  225.5  399.0   539.00  1449.0
N764AS    25.0  398.600000  371.282574  31.0  199.0  261.0   539.00  1449.0
N768AS    24.0  482.708333  269.439108  31.0  381.5  503.0   549.00  1449.0
N795AS    16.0  758.000000  483.892412  31.0  415.5  812.5  1030.50  1721.0


**4.** Из какого аэропорта было произведено больше всего вылетов? В каком городе он находится?

In [107]:
print(data['Origin'].value_counts().idxmax())

ATL


atl - atlanta

**5.** Найдите для каждого аэропорта среднее время полета (`AirTime`) по всем вылетевшим из него рейсам. Какой аэропорт имеет наибольшее значение этого показателя?

In [126]:
meanAirTime = data.groupby('Origin')['AirTime'].mean()
print(meanAirTime)
print(meanAirTime[meanAirTime == meanAirTime.max()])

Origin
ABE    88.266667
ABI    36.400000
ABQ    93.454321
ABY    35.714286
ACK    50.800000
         ...    
WRG    18.000000
XNA    85.945736
YAK    35.900000
YKM    79.000000
YUM    47.470588
Name: AirTime, Length: 297, dtype: float64
Origin
SJU    205.2
Name: AirTime, dtype: float64


**6.** Найдите аэропорт, у которого наибольшая доля задержанных (`DepDelay > 0`) рейсов. Исключите при этом из рассмотрения аэропорты, из которых было отправлено меньше 1000 рейсов (используйте функцию `filter` после `groupby`).

In [201]:
max_depdelay = data.groupby("Origin").filter(lambda x: len(x) >1000).groupby("Origin")['DepDelay'].apply(lambda x: (x>0).sum()).sort_values(ascending=False)
print(max_depdelay.head(1))

Origin
ATL    1739
Name: DepDelay, dtype: int64
